# ACEC-v5 episode 50/75/100 diagnostic

## tl;dr

- **Episode 50 is the current stopping point.** From episode 50 to 100,
  held-out processed EM fell 3.13 points, processed F1 fell 3.27 points,
  Gold supporting-fact recall fell 5.08 points, and retrieval calls fell from
  1.461 to 1.297 per question.
- **The answer regression is mostly semantic replacement, not formatting.**
  Step 100 lost 17 exact answers that step 50 got right and gained only nine;
  11 of the 17 losses had zero F1 after the change. Mean answer length rose
  slightly rather than collapsing.
- **Retrieval compression is real but does not fully explain answer loss.**
  Fourteen of sixteen held-out batches used fewer retrievals at step 100 and
  eleven also lost Gold-SF recall, but batch-level SF changes did not track
  processed-F1 changes closely.
- **The training proxies diverged from fixed held-out behavior.** The final
  10 training episodes had higher mean reward, online EM, and online SF recall
  than the window ending at step 50, but those windows contain different
  sequential training questions and are not a fixed validation set.


## Context & Methods

This diagnostic aligns the same 256 HotpotQA dev questions across ACEC-v5
steps 50, 75, and 100. Answer metrics are available per question. Retrieval
calls and Gold supporting-fact recall were logged only as 16-question batch
aggregates, so retrieval relationships are descriptive at batch grain and
must not be interpreted as per-question causality.

### Key assumptions

- The fixed held-out manifest and all three processed-prediction files have
  one unique row per question and matching ids.
- R3 processed EM/F1 includes the original answer plus extracted candidate
  answers, following the saved evaluator implementation.
- Question-type labels are deterministic lexical heuristics for diagnostic
  slicing, not authoritative HotpotQA categories.


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
summary = json.loads((ROOT / "analysis_results.json").read_text())
per_question = pd.read_csv(ROOT / "per_question_diagnostic.csv")
drivers = pd.read_csv(ROOT / "question_type_drivers.csv")
batches = pd.read_csv(ROOT / "batch_diagnostic.csv")
training = pd.read_csv(ROOT / "training_episode_metrics.csv")
transitions = pd.read_csv(ROOT / "transition_examples.csv")

assert len(per_question) == 256
assert per_question["id"].nunique() == 256
assert len(training) == 100
print("validated: 256 unique held-out questions, 100 training episodes")

validated: 256 unique held-out questions, 100 training episodes


## Results

### Step 50 leads the saved checkpoints on fixed held-out quality

Both direct and processed metrics peak at step 50. The drop is not a product
of one metric: direct F1, processed F1, substring EM, and Gold-SF recall all
move downward by step 100 while retrieval calls continue to fall.


In [2]:
overall = pd.DataFrame(summary["overall"]).T[
    ["direct_em", "direct_f1", "processed_em", "processed_f1",
     "substring_em", "gold_sf_recall", "retrieval_calls", "mean_prediction_words"]
]
overall.round(4)

,direct_em,direct_f1,processed_em,processed_f1,substring_em,gold_sf_recall,retrieval_calls,mean_prediction_words
step100,0.3477,0.4498,0.3633,0.4681,0.3828,0.6816,1.2969,3.4219
step50,0.3672,0.4773,0.3945,0.5008,0.4062,0.7324,1.4609,3.2305
step75,0.3320,0.4562,0.3555,0.4708,0.3828,0.7012,1.4180,3.2891


### Later checkpoints replace correct entities with wrong ones

Step 75 loses 20 processed-exact answers from step 50 and gains 10. Step 100
loses 17 and gains nine. The paired bootstrap intervals cross zero on this
256-question subset, so the ranking is not statistically resolved, but the
point estimates and multiple answer metrics consistently favor step 50.


In [3]:
comparison_rows = []
for name, values in summary["comparisons"].items():
    comparison_rows.append({"comparison": name, **values})
pd.DataFrame(comparison_rows)[
    ["comparison", "processed_em_delta", "processed_em_ci95",
     "processed_f1_delta", "processed_f1_ci95", "wins", "losses",
     "prediction_changed", "lost_exact_with_zero_after_f1"]
]

,comparison,processed_em_delta,processed_em_ci95,processed_f1_delta,processed_f1_ci95,wins,losses,prediction_changed,lost_exact_with_zero_after_f1
0,step50_to_step100,-0.031250,"[-0.0703125, 0.0078125]",-0.032711,"[-0.06994678402161365, 0.004768936228945803]",9,17,115,11
1,step50_to_step75,-0.039062,"[-0.08203125, 0.0]",-0.030011,"[-0.06757550075916459, 0.007126459631055218]",10,20,106,13
2,step75_to_step100,0.007812,"[-0.03125, 0.046875]",-0.002701,"[-0.03987468566972244, 0.03499271102671838]",14,12,100,9


### Most questions are checkpoint-stable; a small unstable set drives the ranking

Across all three checkpoints, 73 questions are always processed-exact and 142
are always wrong. Only 41 questions change exactness at least once. Step 50
already captures 101 of the 114 questions that any of these checkpoints can
answer exactly.


In [4]:
pd.DataFrame(
    [{"pattern (50/75/100)": key, "questions": value}
     for key, value in summary["stability"]["correctness_pattern_counts"].items()]
).sort_values("pattern (50/75/100)")

,pattern (50/75/100),questions
0,000,142
1,001,3
2,010,4
3,011,6
4,100,9
5,101,11
6,110,8
7,111,73


### Who and what questions explain more than the net step-100 F1 loss

`who` and `what` slices contribute approximately -3.57 points to the overall
step100-minus-step50 F1 movement, more than the observed -3.27 points because
`which` and numeric questions partially offset the decline. The answer-type
slice is heuristic, and small slices such as numeric and where should not be
treated as stable estimates.


In [5]:
driver_view = drivers[
    ["question_type", "n", "step50_processed_f1", "step100_processed_f1",
     "step100_vs_step50_f1_delta", "step100_vs_step50_f1_contribution"]
].copy()
driver_view.sort_values("step100_vs_step50_f1_contribution").round(4)

,question_type,n,step50_processed_f1,step100_processed_f1,step100_vs_step50_f1_delta,step100_vs_step50_f1_contribution
2,what,64,0.4434,0.3523,-0.0911,-0.0228
6,who,21,0.5850,0.4272,-0.1578,-0.0129
3,when/year,15,0.4593,0.3667,-0.0926,-0.0054
7,yes/no,18,0.8407,0.7852,-0.0556,-0.0039
4,where,4,0.6667,0.6000,-0.0667,-0.0010
1,other,91,0.4770,0.4794,0.0025,0.0009
5,which,37,0.5118,0.5534,0.0416,0.0060
0,numeric,6,0.0833,0.3611,0.2778,0.0065


### Retrieval compression accompanies lower evidence coverage

From step 50 to 100, retrieval calls decline in 14 of 16 held-out batches;
11 of those batches also lose Gold-SF recall. The batch-level correlation
between retrieval-call change and SF-recall change is 0.58. However, SF-recall
change has near-zero correlation with processed-F1 change at this coarse
grain, so evidence loss is a likely contributor but not a complete answer.


In [6]:
pd.DataFrame(summary["batch_diagnostics"]).T.round(4)

,batches,corr_retrieval_delta_processed_f1_delta,corr_retrieval_delta_sf_delta,corr_sf_delta_processed_f1_delta,retrieval_and_sf_down_batches,retrieval_down_batches,sf_down_batches
step100_vs_step50,16.0,0.0973,0.5812,-0.0572,11.0,14.0,11.0
step75_vs_step50,16.0,0.1746,0.6203,-0.3135,9.0,11.0,11.0


### Online training proxies do not validate the fixed held-out trend

The ten-episode window ending at step 100 has higher online reward, online EM,
and online SF recall than the window ending at step 50, while the fixed
held-out evaluation worsens. Because each training window contains different
sequential questions, this is evidence of proxy/generalization mismatch but
not by itself proof of causal reward hacking.


In [7]:
pd.DataFrame(summary["training_last10_windows"]).T.round(4)

,kl,mean_R,online_em,retrievals,sf_recall
last10_to_step100,0.0961,0.6424,0.5415,1.253,0.7152
last10_to_step25,0.0280,0.4511,0.3649,1.509,0.6410
last10_to_step50,0.0627,0.5512,0.4609,1.430,0.6640
last10_to_step75,0.0658,0.5305,0.4330,1.366,0.6800


## Limitations, uncertainty, and robustness checks

- The fixed held-out set has 256 questions. Step 50's advantage over steps 75
  and 100 is directionally consistent but its paired 95% intervals include
  zero.
- Retrieval diagnostics are limited to 16 batches, not per-question traces.
  The original evaluator discarded individual trajectories and retrieved
  document ids.
- The API extractor was called separately for identical question/prediction
  pairs. Of 914 calls, 282 could have been reused; two duplicate groups
  disagreed on processed EM and five on F1. Caching identical inputs would
  remove this avoidable measurement noise and reduce cost. The noise audit
  does not overturn the step-50 conclusion.


In [8]:
pd.Series(summary["extractor_reuse_audit"], name="value").to_frame()

,value
duplicate_groups,160
duplicate_groups_with_processed_em_disagreement,2
duplicate_groups_with_processed_f1_disagreement,5
extracted_api_calls,914
reusable_calls_if_cached,282
unique_question_prediction_pairs,632


## Recommended next steps

1. Treat step 50 as the current checkpoint and do not extend v5 unchanged to
   500/800 episodes.
2. Save per-question retrieval traces in the next evaluator: queries, actions,
   document ids/titles, Gold-SF hits, and stopping turn.
3. Cache processed-answer extraction by `(question, prediction)` across model
   variants.
4. Build v6 around two protections: calibrate/guard the early-answer decision
   against Gold-SF loss, and anchor the answer generator with stronger KL,
   answer-token masking, or a small supervised rehearsal set.
5. Gate the next run at 25-episode intervals on fixed held-out processed EM/F1,
   semantic ACC, Gold-SF recall, and retrieval calls. Scale toward 500 only if
   quality does not deteriorate while efficiency improves.

## Further questions

- Do the 17 step50-to-step100 exact losses correspond to premature stopping,
  wrong retrieved evidence, or failure to use adequate evidence?
- Can an answer-head anchor preserve entity selection while the action policy
  continues learning lower-cost retrieval?
- Does the step-50 advantage replicate on a larger or second fixed dev sample?
